# MapAid — Entrenamiento del modelo de detección de daños
**Reto 2 — Humanitarian OpenStreetMap Team**

Este notebook entrena una red neuronal (U-Net con encoder ResNet18) para detectar
edificios dañados comparando imágenes de satélite antes y después de un desastre.

**Dataset:** xBD / xView2  
**Tiempo estimado:** 2-3 horas con GPU T4 gratuita de Colab  
**Resultado:** `modelo_danos.pt` — listo para copiar en `backend/ia/`

---
## Antes de empezar
1. Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4**
2. Montar Google Drive con los datos de xBD (ver celda 2)
3. Ejecutar todas las celdas en orden


## 1. Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU disponible. Ve a Entorno de ejecución → "
        "Cambiar tipo → GPU T4 y vuelve a ejecutar."
    )

dispositivo = torch.device("cuda")
print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Montar Google Drive

Sube la carpeta `data/raw/xbd/` de tu proyecto a Google Drive antes de continuar.
La estructura esperada es:
```
Mi unidad/
  mapaid/
    data/raw/xbd/
      images/   ← los .png pre y post
      labels/   ← los .json con etiquetas
```


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
XBD_DIR = "/content/drive/MyDrive/mapaid/data/raw/xbd"
assert os.path.isdir(XBD_DIR), f"No se encuentra {XBD_DIR}"
print(f"✅ Dataset encontrado en {XBD_DIR}")

imagenes = [f for f in os.listdir(f"{XBD_DIR}/images") if f.endswith(".png")]
print(f"   Imágenes: {len(imagenes)} archivos")
print(f"   Escenas: {len(imagenes)//2} pares (pre + post)")


## 3. Instalar dependencias

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
print("✅ Dependencias instaladas")


## 4. Dataset de xBD

Cada muestra es un par de imágenes (pre + post) de 512×512 px.
La etiqueta es una máscara por píxel con 5 clases según la escala oficial de xBD:

| Clase | Valor | Significado |
|---|---|---|
| Sin daño | 0 | Edificio intacto |
| Daño menor | 1 | Grietas, agua alrededor |
| Daño mayor | 2 | Colapso parcial |
| Destruido | 3 | Calcinado o colapsado |
| Sin clasificar | 4 | No determinado |


In [ ]:
import json, os, re
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, random_split

XBD_IMAGES = Path(XBD_DIR) / "images"
XBD_LABELS = Path(XBD_DIR) / "labels"

# Escala de daño xBD → índice de clase
DANO_A_CLASE = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3,
    "un-classified": 4,
}

TAMANO = 512   # px — ResNet18 funciona bien con 512x512
N_CLASES = 5


def wkt_a_mascara(wkt, ancho, alto):
    \"\"\"Convierte un polígono WKT en una máscara binaria (0/1).\"\"\"
    numeros = re.findall(r"-?\d+\.?\d*", wkt)
    pts = np.array(numeros, dtype=float).reshape(-1, 2).astype(np.int32)
    mascara = np.zeros((alto, ancho), dtype=np.uint8)
    cv2.fillPoly(mascara, [pts], 1)
    return mascara


class XBDDataset(Dataset):
    def __init__(self, escenas, transformacion=None):
        self.escenas = escenas
        self.transformacion = transformacion

    def __len__(self):
        return len(self.escenas)

    def __getitem__(self, idx):
        base = self.escenas[idx]
        img_pre  = cv2.imread(str(XBD_IMAGES / f"{base}_pre_disaster.png"))
        img_post = cv2.imread(str(XBD_IMAGES / f"{base}_post_disaster.png"))
        img_pre  = cv2.cvtColor(img_pre,  cv2.COLOR_BGR2RGB)
        img_post = cv2.cvtColor(img_post, cv2.COLOR_BGR2RGB)

        # Concatenar pre y post como 6 canales (entrada a la red)
        entrada = np.concatenate([img_pre, img_post], axis=2)

        # Construir máscara de clases a partir de las etiquetas JSON
        etiquetas = json.loads(
            (XBD_LABELS / f"{base}_post_disaster.json").read_text()
        )
        alto, ancho = img_post.shape[:2]
        mascara = np.zeros((alto, ancho), dtype=np.int64)

        for feat in etiquetas.get("features", {}).get("xy", []):
            clase = DANO_A_CLASE.get(
                feat["properties"].get("subtype", "un-classified"), 4
            )
            if clase == 0:
                continue  # Los intactos no se anotan en la máscara
            m = wkt_a_mascara(feat["wkt"], ancho, alto)
            mascara[m == 1] = clase

        if self.transformacion:
            aug = self.transformacion(image=entrada, mask=mascara)
            entrada, mascara = aug["image"], aug["mask"]

        return entrada.float() / 255.0, mascara.long()


# Buscar todos los pares completos
pares = sorted([
    f.stem.replace("_post_disaster", "")
    for f in XBD_IMAGES.glob("*_post_disaster.png")
    if (XBD_IMAGES / f.stem.replace("post", "pre")).with_suffix(".png").exists()
    and (XBD_LABELS / f.stem).with_suffix(".json").exists()
])
print(f"✅ {len(pares)} pares completos encontrados")

# Aumentaciones de entrenamiento
transform_train = A.Compose([
    A.RandomCrop(TAMANO, TAMANO),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.Rotate(limit=15, p=0.3),
    ToTensorV2(),
])
transform_val = A.Compose([A.CenterCrop(TAMANO, TAMANO), ToTensorV2()])

# Split 80/20
n_val = max(1, int(len(pares) * 0.2))
n_train = len(pares) - n_val
ds_train = XBDDataset(pares[:n_train], transform_train)
ds_val   = XBDDataset(pares[n_train:], transform_val)

BATCH = 4
dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2)
print(f"   Train: {len(ds_train)} muestras | Val: {len(ds_val)} muestras")


## 5. Arquitectura del modelo

**U-Net con encoder ResNet18** preentrenado en ImageNet.

- **Encoder:** ResNet18 — extrae características de las imágenes
- **Decoder:** U-Net — reconstruye la máscara a la resolución original
- **Entrada:** 6 canales (3 del pre + 3 del post)
- **Salida:** máscara de 5 clases por píxel

El encoder ResNet18 preentrenado en ImageNet acelera mucho el entrenamiento:
en vez de aprender desde cero qué es una textura o un borde, ya lo sabe
y solo tiene que aprender a reconocer daños en imágenes de satélite.


In [ ]:
import segmentation_models_pytorch as smp

modelo = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=6,        # pre + post concatenados
    classes=N_CLASES,
    activation=None,      # usamos CrossEntropyLoss, que incluye softmax
)
modelo = modelo.to(dispositivo)

n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"✅ Modelo creado: {n_params:,} parámetros entrenables")


## 6. Entrenamiento

**Función de pérdida:** CrossEntropy con pesos por clase.
Los edificios dañados son mucho menos frecuentes que los intactos,
así que les damos más peso para que el modelo no los ignore.

**Optimizador:** AdamW con scheduler coseno — empieza con lr alto
y lo baja suavemente para afinar al final.


In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Pesos de clase: más peso a los dañados para compensar el desequilibrio
PESOS_CLASE = torch.tensor([0.5, 2.0, 3.0, 4.0, 1.0]).to(dispositivo)
criterio = nn.CrossEntropyLoss(weight=PESOS_CLASE, ignore_index=-1)

optimizador = AdamW(modelo.parameters(), lr=1e-4, weight_decay=1e-4)
N_EPOCAS = 30
scheduler = CosineAnnealingLR(optimizador, T_max=N_EPOCAS, eta_min=1e-6)


def iou_por_clase(pred, target, n_clases=N_CLASES):
    \"\"\"Intersection over Union por clase — métrica principal de xBD.\"\"\"
    ious = []
    pred_flat = pred.argmax(1).view(-1)
    target_flat = target.view(-1)
    for c in range(1, n_clases):  # ignorar clase 0 (sin daño)
        inter = ((pred_flat == c) & (target_flat == c)).sum().float()
        union = ((pred_flat == c) | (target_flat == c)).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return np.mean(ious) if ious else 0.0


mejor_iou = 0.0
historial = {"train_loss": [], "val_loss": [], "val_iou": []}

for epoca in range(N_EPOCAS):
    # ── Train ──
    modelo.train()
    perdida_train = 0.0
    for imagenes, mascaras in dl_train:
        imagenes = imagenes.to(dispositivo)
        mascaras = mascaras.to(dispositivo)
        optimizador.zero_grad()
        pred = modelo(imagenes)
        loss = criterio(pred, mascaras)
        loss.backward()
        optimizador.step()
        perdida_train += loss.item()
    perdida_train /= len(dl_train)

    # ── Validación ──
    modelo.eval()
    perdida_val = 0.0
    ious = []
    with torch.no_grad():
        for imagenes, mascaras in dl_val:
            imagenes = imagenes.to(dispositivo)
            mascaras = mascaras.to(dispositivo)
            pred = modelo(imagenes)
            perdida_val += criterio(pred, mascaras).item()
            ious.append(iou_por_clase(pred, mascaras))
    perdida_val /= len(dl_val)
    val_iou = np.mean(ious)

    scheduler.step()
    historial["train_loss"].append(perdida_train)
    historial["val_loss"].append(perdida_val)
    historial["val_iou"].append(val_iou)

    # Guardar el mejor modelo
    if val_iou > mejor_iou:
        mejor_iou = val_iou
        torch.save(modelo.state_dict(), "modelo_danos.pt")
        estrella = " ⭐ guardado"
    else:
        estrella = ""

    print(
        f"Época {epoca+1:02d}/{N_EPOCAS} | "
        f"loss train {perdida_train:.4f} | "
        f"loss val {perdida_val:.4f} | "
        f"IoU val {val_iou:.4f}{estrella}"
    )

print(f"\n✅ Entrenamiento completado. Mejor IoU: {mejor_iou:.4f}")


## 7. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(historial["train_loss"], label="Train")
ax1.plot(historial["val_loss"],   label="Val")
ax1.set_title("Pérdida (CrossEntropy)")
ax1.set_xlabel("Época"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(historial["val_iou"], color="green")
ax2.set_title("IoU de validación (clases dañadas)")
ax2.set_xlabel("Época"); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("curvas_entrenamiento.png", dpi=150)
plt.show()
print(f"Mejor IoU alcanzado: {max(historial['val_iou']):.4f}")


## 8. Visualizar predicciones sobre ejemplos del conjunto de validación

In [ ]:
import matplotlib.patches as mpatches

COLORES = {0:"#cccccc", 1:"#f0c419", 2:"#e8871a", 3:"#c0392b", 4:"#888888"}
NOMBRES = {0:"Sin daño", 1:"Daño menor", 2:"Daño mayor", 3:"Destruido", 4:"Sin clasificar"}

modelo.eval()
muestra_imgs, muestra_masks = next(iter(dl_val))
with torch.no_grad():
    pred = modelo(muestra_imgs.to(dispositivo)).argmax(1).cpu()

fig, axes = plt.subplots(min(4, BATCH), 4, figsize=(16, min(4,BATCH)*4))
if min(4,BATCH) == 1:
    axes = [axes]

for i in range(min(4, BATCH)):
    pre  = muestra_imgs[i, :3].permute(1,2,0).numpy()
    post = muestra_imgs[i, 3:].permute(1,2,0).numpy()

    def colorear(mascara):
        rgb = np.zeros((*mascara.shape, 3), dtype=np.uint8)
        for cls, hex_color in COLORES.items():
            r,g,b = int(hex_color[1:3],16), int(hex_color[3:5],16), int(hex_color[5:7],16)
            rgb[mascara==cls] = [r,g,b]
        return rgb

    axes[i][0].imshow(np.clip(pre,0,1));  axes[i][0].set_title("Antes"); axes[i][0].axis("off")
    axes[i][1].imshow(np.clip(post,0,1)); axes[i][1].set_title("Después"); axes[i][1].axis("off")
    axes[i][2].imshow(colorear(muestra_masks[i].numpy())); axes[i][2].set_title("Real"); axes[i][2].axis("off")
    axes[i][3].imshow(colorear(pred[i].numpy())); axes[i][3].set_title("Predicción"); axes[i][3].axis("off")

parches = [mpatches.Patch(color=COLORES[c], label=NOMBRES[c]) for c in range(N_CLASES)]
fig.legend(handles=parches, loc="lower center", ncol=5, bbox_to_anchor=(0.5,-0.02))
plt.tight_layout()
plt.savefig("predicciones.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Descargar el modelo entrenado

Descarga `modelo_danos.pt` y cópialo en `backend/ia/` de tu proyecto.

El backend lo cargará automáticamente en vez del comparador de píxeles actual.


In [ ]:
from google.colab import files

# Descargar el modelo
files.download("modelo_danos.pt")
files.download("curvas_entrenamiento.png")
files.download("predicciones.png")

print("✅ Archivos descargados.")
print()
print("Próximo paso:")
print("  1. Copia modelo_danos.pt en backend/ia/")
print("  2. Reinicia el backend (python -m uvicorn main:app --reload)")
print("  3. El comparador usará automáticamente el modelo entrenado")
